# Lesson 04 - 도구 사용 디자인 패턴

이 강의에서는 Microsoft Agent Framework (Python)를 사용하여 AI 에이전트를 위한 **도구 사용** 디자인 패턴을 학습합니다. 다음 내용을 다룹니다:

- `@tool` 데코레이터와 타입이 지정된 매개변수를 사용하여 함수 도구 정의하기
- 모델이 각 도구의 기능을 알도록 도구 스키마 제공하기
- `approval_mode`로 도구 실행 제어하기
- Pydantic 모델과 `response_format`을 통해 **구조화된 출력** 반환하기

시나리오는 목적지를 조회하고 가용성을 확인하며 항공편 정보를 검색할 수 있는 **여행 예약 에이전트**입니다.


## 설정


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## @tool 데코레이터로 도구 정의하기

`@tool` 데코레이터는 일반 Python 함수를 에이전트가 호출할 수 있는 도구로 바꿉니다.  
핵심 사항:

- **도큐스트링**이 모델이 보는 도구 설명이 됩니다.  
- **타입 주석**(설명이 포함된 `Annotated` 포함)이 도구 스키마를 정의합니다.  
- `approval_mode`는 실행 전에 사용자가 각 호출을 승인해야 하는지를 제어합니다.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get available vacation destinations."""
    return ["Barcelona", "Paris", "Berlin", "Tokyo", "Sydney", "New York City"]


@tool(approval_mode="never_require")
def check_availability(
    destination: Annotated[str, "The destination to check"],
) -> str:
    """Check booking availability for a destination."""
    availability = {
        "Barcelona": "Available - 3 spots left",
        "Paris": "Available",
        "Berlin": "Sold out",
        "Tokyo": "Available - 1 spot left",
        "Sydney": "Available",
        "New York City": "Available",
    }
    return availability.get(destination, "Unknown destination")


@tool(approval_mode="never_require")
def get_flight_info(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
) -> str:
    """Get flight information between two cities."""
    flights = {
        "LHR-BCN": "BA 2042, Departs 08:30, Arrives 11:45, $350",
        "LHR-CDG": "AF 1081, Departs 09:15, Arrives 11:30, $280",
        "LHR-NRT": "JL 044, Departs 11:00, Arrives 07:00+1, $890",
    }
    return flights.get(
        f"{origin}-{destination}",
        f"No direct flights from {origin} to {destination}",
    )

## 여러 도구를 사용하는 에이전트 생성

세 가지 도구 모두를 클라이언트에 전달하여 모델이 사용자의 질문에 답변하는 데 필요한 도구를 호출할 수 있도록 합니다.


In [ ]:
travel_tools = [get_destinations, check_availability, get_flight_info]

agent = await provider.create_agent(
    name="TravelToolAgent",
    instructions="You are a travel agent. Use the available tools to answer questions about destinations, availability, and flights.",
    tools=travel_tools,
)

response = await agent.run(
    "What destinations do you have? Which ones are still available?"
)
print(response)

## 도구를 사용한 구조화된 출력

`response_format`을 Pydantic 모델로 설정하면 에이전트가 자유 형식 텍스트 대신 잘 형식화된 JSON 객체를 반환하도록 강제할 수 있습니다. 이는 하위 코드가 결과를 프로그래밍 방식으로 처리해야 할 때 유용합니다.


In [ ]:
class BookingRecommendation(BaseModel):
    destination: str
    available: bool
    flight_details: str
    estimated_cost: int


class TravelPlan(BaseModel):
    recommendations: list[BookingRecommendation]


structured_agent = await provider.create_agent(
    name="StructuredTravelAgent",
    instructions=(
        "You are a travel agent. Use the available tools to find destinations, "
        "check availability, and get flight info. Return structured results."
    ),
    tools=[get_destinations, check_availability, get_flight_info],
)

response = await structured_agent.run(
    "I want to fly from London Heathrow to somewhere warm in Europe. "
    "Check what's available."
)
if response:
    print(response)

## 도구 승인 패턴

`@tool`의 `approval_mode` 매개변수는 도구 호출이 실행되기 전에 인간의 승인을 요구하는지 여부를 제어합니다:

| 모드 | 동작 |
|---|---|
| `"never_require"` | 도구가 자동으로 실행됨 — 사용자 확인 불필요. |
| `"always_require"` | 모든 호출은 실행 전에 사용자의 승인을 받아야 함. |

부작용이 있는 도구(예: 비행기 예약, 신용카드 결제)에는 `"always_require"`를 사용하여 사람이 개입하도록 합니다.


In [ ]:
@tool(approval_mode="always_require")
def book_flight(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
    passenger_name: Annotated[str, "Full name of the passenger"],
) -> str:
    """Book a flight for a passenger. Requires approval before executing."""
    return (
        f"Flight booked from {origin} to {destination} "
        f"for {passenger_name}. Confirmation #TRV-2024-{hash(passenger_name) % 10000:04d}"
    )


print("Tool name:", book_flight.name)
print("Approval mode:", book_flight.approval_mode)

## 요약

이 수업에서 배운 내용은 다음과 같습니다:

1. `@tool` 데코레이터를 사용하여 도구를 정의하고, 도구 스키마 역할을 하는 타입이 지정된 매개변수와 도큐스트링을 작성하는 방법
2. 여러 도구를 조합하여 에이전트가 복잡한 쿼리에 답변하기 위해 순차적으로 호출할 수 있도록 하는 방법
3. Pydantic 모델을 `response_format`으로 전달하여 구조화된 출력을 반환하는 방법
4. `approval_mode`를 사용하여 민감한 작업에 대해 인간이 개입하도록 도구 승인 제어 방법

이러한 패턴은 외부 시스템과 안전하게 상호작용할 수 있는 신뢰성 있고 실무에 적합한 에이전트를 구축하는 기초가 됩니다.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**면책 조항**:  
이 문서는 AI 번역 서비스 [Co-op Translator](https://github.com/Azure/co-op-translator)를 이용하여 번역되었습니다. 정확성을 위해 최선을 다하고 있으나, 자동 번역에는 오류나 부정확한 부분이 있을 수 있음을 유의하시기 바랍니다. 원문 문서가 권위 있는 출처로 간주되어야 합니다. 중요한 정보의 경우에는 전문적인 인간 번역을 권장합니다. 이 번역본 사용으로 인한 오해나 잘못된 해석에 대해 당사는 책임을 지지 않습니다.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
